# CUDA Streams

Design an experiment for asynchronous CUDA execution and overlap.

## Objectives

Distinguish stream ordering, host asynchrony, synchronization, and genuine overlap.

## Background

CUDA streams order operations within a stream while potentially allowing independent work in different streams to overlap.

## Prediction

CUDA operations issued from Python are normally asynchronous with respect to the host. A Python call can return after work has been placed into a CUDA stream, before the GPU has completed that work.

For a sequence of matrix multiplications in one CUDA stream, we should therefore distinguish three measurements:

1. **host enqueue time**: how long Python takes to submit the operations;
2. **GPU elapsed time**: how long the operations take on the GPU, measured with CUDA events;
3. **synchronized wall time**: how long the host observes from the first submission until all submitted work has completed.

Because all matrix multiplications are submitted to the same stream, they should execute in issue order. The host enqueue time should be substantially shorter than the synchronized wall time when the queued GPU workload is large enough.

Immediately after submission, the final CUDA event may still be incomplete. Waiting for that event should account for most of the difference between enqueue time and synchronized wall time.

CUDA-event time and host wall time need not be identical. They use different clocks and include different boundaries:

- CUDA events measure elapsed device work between two positions in a stream;
- synchronized wall time also includes Python submission overhead and the host's synchronization call.

This first experiment establishes timing and ordering semantics only. It does not demonstrate concurrent execution or overlap between streams.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [2]:
from pprint import pprint

import torch

from common.cuda import detect_cuda


cuda_info = detect_cuda()
pprint(cuda_info)

if not cuda_info.torch_installed:
    raise RuntimeError(
        "PyTorch is not installed in this environment. "
        "CUDA stream experiments cannot continue."
    )

if not cuda_info.available:
    raise RuntimeError(
        "PyTorch is installed, but CUDA is not available. "
        f"Detection error: {cuda_info.error!r}"
    )

device_index = torch.cuda.current_device()
device = torch.device(f"cuda:{device_index}")
device_properties = torch.cuda.get_device_properties(device_index)

stream_environment = {
    "device_index": device_index,
    "device_name": torch.cuda.get_device_name(device_index),
    "compute_capability": (
        device_properties.major,
        device_properties.minor,
    ),
    "multiprocessor_count": device_properties.multi_processor_count,
    "default_stream": str(torch.cuda.default_stream(device)),
}

pprint(stream_environment)

CudaInfo(torch_installed=True,
         available=True,
         device_count=1,
         device_names=('NVIDIA GB10',),
         torch_version='2.13.0+cu130',
         cuda_version='13.0',
         error=None)
{'compute_capability': (12, 1),
 'default_stream': '<torch.cuda.Stream device=cuda:0 cuda_stream=0x0>',
 'device_index': 0,
 'device_name': 'NVIDIA GB10',
 'multiprocessor_count': 48}


### Host submission versus GPU completion

The default CUDA stream orders operations submitted to it. The following experiment submits a sequence of matrix multiplications to that stream.

Before each measured trial, the host synchronizes with the device. This prevents unfinished work from an earlier trial from contaminating the measurement.

Two CUDA events bracket the matrix multiplications:

- the start event is recorded before the first multiplication;
- the end event is recorded after the final multiplication.

Recording an event is itself asynchronous. Calling `query()` on the end event immediately after submission tells us whether the GPU has already reached that point without forcing synchronization.

The experiment then waits for the end event and reports:

- Python submission time;
- additional time spent waiting;
- total synchronized wall time;
- CUDA-event elapsed time.

The workload reuses preallocated output storage so allocation is not part of each matrix multiplication.

In [3]:
from time import perf_counter

import pandas as pd


MATRIX_SIZE = 4096
MATRIX_MULTIPLICATIONS = 32
DTYPE = torch.float32

generator = torch.Generator(device=device)
generator.manual_seed(20260802)

left = torch.randn(
    (MATRIX_SIZE, MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=generator,
)
right = torch.randn(
    (MATRIX_SIZE, MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=generator,
)
result = torch.empty_like(left)

matrix_bytes = left.numel() * left.element_size()
operation_flops = 2 * MATRIX_SIZE**3

workload_info = {
    "matrix_size": MATRIX_SIZE,
    "dtype": str(DTYPE),
    "bytes_per_matrix_mib": matrix_bytes / 1024**2,
    "matrix_multiplications": MATRIX_MULTIPLICATIONS,
    "flops_per_multiplication": operation_flops,
    "total_submitted_flops": operation_flops * MATRIX_MULTIPLICATIONS,
}

pprint(workload_info)

{'bytes_per_matrix_mib': 64.0,
 'dtype': 'torch.float32',
 'flops_per_multiplication': 137438953472,
 'matrix_multiplications': 32,
 'matrix_size': 4096,
 'total_submitted_flops': 4398046511104}


In [4]:
WARMUP_MULTIPLICATIONS = 4

for _ in range(WARMUP_MULTIPLICATIONS):
    torch.mm(left, right, out=result)

torch.cuda.synchronize(device)

print(
    f"Completed {WARMUP_MULTIPLICATIONS} warm-up matrix multiplications "
    "and synchronized the device."
)

Completed 4 warm-up matrix multiplications and synchronized the device.


In [ ]:
def measure_default_stream_trial(trial: int) -> dict[str, float | int | bool]:
    torch.cuda.synchronize(device)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    wall_start = perf_counter()

    start_event.record()

    for _ in range(MATRIX_MULTIPLICATIONS):
        torch.mm(left, right, out=result)

    end_event.record()

    enqueue_end = perf_counter()
    complete_immediately_after_enqueue = end_event.query()

    end_event.synchronize()
    wall_end = perf_counter()

    enqueue_ms = (enqueue_end - wall_start) * 1_000
    synchronized_wall_ms = (wall_end - wall_start) * 1_000
    synchronization_wait_ms = (wall_end - enqueue_end) * 1_000
    gpu_elapsed_ms = start_event.elapsed_time(end_event)

    return {
        "trial": trial,
        "enqueue_ms": enqueue_ms,
        "synchronization_wait_ms": synchronization_wait_ms,
        "synchronized_wall_ms": synchronized_wall_ms,
        "gpu_elapsed_ms": gpu_elapsed_ms,
        "complete_after_enqueue": complete_immediately_after_enqueue,
        "enqueue_fraction": enqueue_ms / synchronized_wall_ms,
    }


TRIALS = 7

timing_results = pd.DataFrame(
    measure_default_stream_trial(trial) for trial in range(1, TRIALS + 1)
)

timing_results.round(
    {
        "enqueue_ms": 3,
        "synchronization_wait_ms": 3,
        "synchronized_wall_ms": 3,
        "gpu_elapsed_ms": 3,
        "enqueue_fraction": 4,
    }
)

,trial,enqueue_ms,synchronization_wait_ms,synchronized_wall_ms,gpu_elapsed_ms,complete_after_enqueue,enqueue_fraction
0,1,1.680,244.131,245.810,245.675,False,0.0068
1,2,0.160,252.753,252.913,252.899,False,0.0006
2,3,0.153,253.287,253.440,253.430,False,0.0006
3,4,0.151,254.116,254.268,254.258,False,0.0006
4,5,0.150,254.351,254.500,254.491,False,0.0006
5,6,0.152,252.129,252.281,252.271,False,0.0006
6,7,0.150,250.903,251.053,251.044,False,0.0006


In [ ]:
timing_summary = (
    timing_results[
        [
            "enqueue_ms",
            "synchronization_wait_ms",
            "synchronized_wall_ms",
            "gpu_elapsed_ms",
            "enqueue_fraction",
        ]
    ]
    .agg(["median", "min", "max"])
    .T
)

completion_counts = (
    timing_results["complete_after_enqueue"]
    .value_counts(dropna=False)
    .rename_axis("complete_after_enqueue")
    .to_frame("trials")
)

display(timing_summary.round(4))
display(completion_counts)

median_enqueue_ms = timing_results["enqueue_ms"].median()
median_wait_ms = timing_results["synchronization_wait_ms"].median()
median_wall_ms = timing_results["synchronized_wall_ms"].median()
median_gpu_ms = timing_results["gpu_elapsed_ms"].median()

print(f"Median host enqueue time: {median_enqueue_ms:.3f} ms")
print(f"Median synchronization wait: {median_wait_ms:.3f} ms")
print(f"Median synchronized wall time: {median_wall_ms:.3f} ms")
print(f"Median CUDA-event GPU time: {median_gpu_ms:.3f} ms")
print(f"Median wall/event difference: {median_wall_ms - median_gpu_ms:.3f} ms")

,median,min,max
enqueue_ms,0.1517,0.1496,1.6795
synchronization_wait_ms,252.7532,244.1309,254.3506
synchronized_wall_ms,252.9130,245.8104,254.5001
gpu_elapsed_ms,252.8991,245.6749,254.4906
enqueue_fraction,0.0006,0.0006,0.0068


,trials
complete_after_enqueue,
False,7


Median host enqueue time: 0.152 ms
Median synchronization wait: 252.753 ms
Median synchronized wall time: 252.913 ms
Median CUDA-event GPU time: 252.899 ms
Median wall/event difference: 0.014 ms


### Independent work in one stream versus two streams

CUDA streams provide independent ordering domains. Operations in separate streams may execute concurrently when:

1. no dependency forces an ordering relationship;
2. the GPU can schedule both workloads concurrently;
3. the workloads do not individually consume all relevant execution resources.

The following experiment keeps the total arithmetic work constant:

- the single-stream case submits all matrix multiplications to one stream;
- the two-stream case submits half to each of two independent streams.

Each stream receives distinct input and output tensors. This avoids data hazards between streams.

A coordinating start event gives both worker streams the same logical release point. Each worker stream records a completion event after its final multiplication. A coordinating stream waits for both completion events before recording the overall end event.

The primary measurement is therefore the complete GPU makespan from coordinated release until both streams have finished.

A large `4096 × 4096` FP32 matrix multiplication is expected to occupy a substantial fraction of the GB10's compute resources. Separate streams make concurrent scheduling possible, but they may not produce a speedup if one multiplication already saturates the relevant hardware.

Prediction:

- both variants should perform the same total number of matrix multiplications;
- two-stream submission may remain inexpensive for the host;
- two streams may show little or no makespan reduction for this large GEMM workload;
- a lack of speedup would not prove that streams cannot overlap smaller or complementary operations.

In [7]:
worker_stream_a = torch.cuda.Stream(device=device)
worker_stream_b = torch.cuda.Stream(device=device)
coordination_stream = torch.cuda.Stream(device=device)

left_a = left
right_a = right
result_a = result

left_b = torch.randn(
    (MATRIX_SIZE, MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=generator,
)
right_b = torch.randn(
    (MATRIX_SIZE, MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=generator,
)
result_b = torch.empty_like(left_b)

if MATRIX_MULTIPLICATIONS % 2 != 0:
    raise ValueError(
        "MATRIX_MULTIPLICATIONS must be even for an equal two-stream split"
    )

MULTIPLICATIONS_PER_STREAM = MATRIX_MULTIPLICATIONS // 2

stream_workload_info = {
    "total_matrix_multiplications": MATRIX_MULTIPLICATIONS,
    "single_stream_multiplications": MATRIX_MULTIPLICATIONS,
    "two_stream_multiplications_per_stream": MULTIPLICATIONS_PER_STREAM,
    "two_stream_total_multiplications": 2 * MULTIPLICATIONS_PER_STREAM,
}

pprint(stream_workload_info)

{'single_stream_multiplications': 32,
 'total_matrix_multiplications': 32,
 'two_stream_multiplications_per_stream': 16,
 'two_stream_total_multiplications': 32}


In [8]:
with torch.cuda.stream(worker_stream_a):
    torch.mm(left_a, right_a, out=result_a)

with torch.cuda.stream(worker_stream_b):
    torch.mm(left_b, right_b, out=result_b)

torch.cuda.synchronize(device)

print("Warmed both worker streams and synchronized the device.")

Warmed both worker streams and synchronized the device.


In [11]:
def measure_single_stream_makespan(
    trial: int,
) -> dict[str, float | int | str]:
    torch.cuda.synchronize(device)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    wall_start = perf_counter()

    with torch.cuda.stream(worker_stream_a):
        start_event.record()

        for _ in range(MATRIX_MULTIPLICATIONS):
            torch.mm(left_a, right_a, out=result_a)

        end_event.record()

    enqueue_end = perf_counter()

    end_event.synchronize()
    wall_end = perf_counter()

    return {
        "trial": trial,
        "configuration": "one stream",
        "enqueue_ms": (enqueue_end - wall_start) * 1_000,
        "synchronized_wall_ms": (wall_end - wall_start) * 1_000,
        "gpu_makespan_ms": start_event.elapsed_time(end_event),
    }


def measure_two_stream_makespan(
    trial: int,
) -> dict[str, float | int | str]:
    torch.cuda.synchronize(device)

    release_event = torch.cuda.Event(enable_timing=True)
    stream_a_done = torch.cuda.Event()
    stream_b_done = torch.cuda.Event()
    makespan_end = torch.cuda.Event(enable_timing=True)

    wall_start = perf_counter()

    with torch.cuda.stream(coordination_stream):
        release_event.record()

    worker_stream_a.wait_event(release_event)
    worker_stream_b.wait_event(release_event)

    with torch.cuda.stream(worker_stream_a):
        for _ in range(MULTIPLICATIONS_PER_STREAM):
            torch.mm(left_a, right_a, out=result_a)

        stream_a_done.record()

    with torch.cuda.stream(worker_stream_b):
        for _ in range(MULTIPLICATIONS_PER_STREAM):
            torch.mm(left_b, right_b, out=result_b)

        stream_b_done.record()

    coordination_stream.wait_event(stream_a_done)
    coordination_stream.wait_event(stream_b_done)

    with torch.cuda.stream(coordination_stream):
        makespan_end.record()

    enqueue_end = perf_counter()

    makespan_end.synchronize()
    wall_end = perf_counter()

    return {
        "trial": trial,
        "configuration": "two streams",
        "enqueue_ms": (enqueue_end - wall_start) * 1_000,
        "synchronized_wall_ms": (wall_end - wall_start) * 1_000,
        "gpu_makespan_ms": release_event.elapsed_time(makespan_end),
    }

In [12]:
STREAM_TRIALS = 7

stream_measurements: list[dict[str, float | int | str]] = []

for trial in range(1, STREAM_TRIALS + 1):
    if trial % 2 == 1:
        stream_measurements.append(measure_single_stream_makespan(trial))
        stream_measurements.append(measure_two_stream_makespan(trial))
    else:
        stream_measurements.append(measure_two_stream_makespan(trial))
        stream_measurements.append(measure_single_stream_makespan(trial))

stream_results = pd.DataFrame(stream_measurements)

stream_results.round(
    {
        "enqueue_ms": 3,
        "synchronized_wall_ms": 3,
        "gpu_makespan_ms": 3,
    }
)

,trial,configuration,enqueue_ms,synchronized_wall_ms,gpu_makespan_ms
0,1,one stream,1.492,247.108,246.931
1,1,two streams,0.222,248.967,248.940
2,2,two streams,0.185,248.428,248.413
3,2,one stream,0.155,253.628,253.614
4,3,one stream,0.156,253.096,253.082
5,3,two streams,0.181,248.639,248.624
6,4,two streams,0.179,249.638,249.624
7,4,one stream,0.156,253.938,253.924
8,5,one stream,0.158,255.282,255.268
9,5,two streams,0.180,249.657,249.643


In [ ]:
stream_summary = stream_results.groupby("configuration")[
    [
        "enqueue_ms",
        "synchronized_wall_ms",
        "gpu_makespan_ms",
    ]
].agg(["median", "min", "max"])

display(stream_summary.round(3))

median_makespans = stream_results.groupby("configuration")["gpu_makespan_ms"].median()

one_stream_ms = median_makespans["one stream"]
two_stream_ms = median_makespans["two streams"]

two_stream_speedup = one_stream_ms / two_stream_ms
makespan_change_percent = (two_stream_ms - one_stream_ms) / one_stream_ms * 100

comparison = pd.DataFrame(
    [
        {
            "one_stream_median_ms": one_stream_ms,
            "two_stream_median_ms": two_stream_ms,
            "two_stream_speedup": two_stream_speedup,
            "two_stream_makespan_change_percent": makespan_change_percent,
        }
    ]
)

display(comparison.round(4))

enqueue_ms               synchronized_wall_ms                    \
                  median    min    max               median      min      max   
configuration                                                                   
one stream         0.156  0.154  1.492              253.924  247.108  255.282   
two streams        0.181  0.179  0.222              249.638  248.428  249.874   

              gpu_makespan_ms                    
                       median      min      max  
configuration                                    
one stream            253.910  246.931  255.268  
two streams           249.624  248.413  249.861

,one_stream_median_ms,two_stream_median_ms,two_stream_speedup,two_stream_makespan_change_percent
0,253.9099,249.6245,1.0172,-1.6878


## Observations

TODO: Record timelines and synchronized elapsed times from actual runs.

## Explanation

TODO: Explain ordering, dependencies, resource contention, and observed overlap.

## Connection to LLMs

Streams can coordinate transfers, kernels, and independent request work in latency-sensitive inference systems.

## Further Exploration

TODO: Vary operation size and dependencies to find when overlap becomes beneficial.